In [1]:
from transformers import pipeline
import glob
import os
import torch
from tqdm import tqdm
import pandas as pd
import re

# Check if GPU is available
device = 0 if torch.cuda.is_available() else -1

# Initializing NLP model from HuggingFace
pipe = pipeline("text-classification", model="ElKulako/cryptobert", device=device, max_length=512, truncation=True, top_k=3)



# Getting input folder and files
input_folder = '/Users/gabo/Documents/Thesis/OpenAI/Testing/CSV/*'
file_list = glob.glob(input_folder)

# Loop through all files in the input folder
for file_path in file_list:
    # Extract the file name
    file_name = os.path.basename(file_path)
    file_name_short = os.path.splitext(file_name)[0]

    # Define the output file path
    output_file_path = f'/Users/gabo/Documents/Thesis/OpenAI/Testing/CSV/{file_name_short}_cryptobert.csv'

    # Check if the output file already exists
    if os.path.exists(output_file_path):
        print(f"Skipped file {file_name}, output file already exists: {output_file_path}")
        continue  # Skip to the next file

    # Skip empty files by checking file size
    if os.path.getsize(file_path) == 0:
        print(f"Skipped file {file_name}, it is empty.")
        continue

    # Load the CSV file
    try:
        df = pd.read_csv(file_path)
    except pd.errors.EmptyDataError:
        print(f"Skipped file {file_name}, no data to parse.")
        continue

    comments_list = df.comment.tolist()

    # Create empty lists to store sentiment scores
    positive_scores = []
    negative_scores = []
    neutral_scores = []

    # Iterate over comments with tqdm for progress bar
    for comment in tqdm(comments_list, desc=f"Processing {file_name}"):
        sentiment = pipe(comment)  # Calling the model to get sentiment prediction

        # Access the nested list structure and extract the sentiment scores
        sentiment_data = sentiment[0]  # This is the list containing the dictionaries

        # Initialize the scores to 0
        positive_score = 0
        negative_score = 0
        neutral_score = 0

        # Loop through the sentiment list (it should contain dictionaries with 'label' and 'score')
        for pred in sentiment_data:
            if pred['label'] == 'Bullish':
                positive_score = pred['score']
            elif pred['label'] == 'Bearish':
                negative_score = pred['score']
            elif pred['label'] == 'Neutral':
                neutral_score = pred['score']

        # Append the scores to their respective lists
        positive_scores.append(positive_score)
        negative_scores.append(negative_score)
        neutral_scores.append(neutral_score)

    # Add the sentiment scores to the DataFrame
    df['positive_score'] = positive_scores
    df['negative_score'] = negative_scores
    df['neutral_score'] = neutral_scores

    # Save the DataFrame with the new sentiment score columns to a CSV file
    df.to_csv(output_file_path, index=False)

    print(f"Processed and saved: {output_file_path}")


Processing testing_data_output.csv: 100%|███| 2331/2331 [01:18<00:00, 29.79it/s]

Processed and saved: /Users/gabo/Documents/Thesis/OpenAI/Testing/CSV/testing_data_output_cryptobert.csv
